In [ ]:
import pandas as pd
import numpy as np
from src_RF_DT import *

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from matplotlib import pyplot as plt

# 1.0 - Classificação de Presença Baseado em Fatores Socioeconômicos Usando Random Forest

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [ ]:
df = pre_processor_rf_dt(df, modelo='DT', n_samples = 1_000_000)

## 1.2 - Construção da Matriz X e Vetor y

In [4]:
X = df.drop([ 'FALTOU'], axis=1)

y = df['FALTOU']

## 1.3 - Separação em Dados de Treino e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1.4 Treinando com os Melhores Parâmetros

In [7]:
cv_rf = tune_random_forest(X_train, y_train, n_iter=10, cv=3, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)
print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  cv_rf.predict(X_test))))
print(classification_report(y_test, cv_rf.predict(X_test)))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
RandomForestClassifier(class_weight='balanced', max_depth=40,
                       max_features='log2', min_samples_leaf=10,
                       min_samples_split=10, random_state=42)
Ein:  0.2853
Eout: 0.3206
              precision    recall  f1-score   support

           0       0.84      0.69      0.76     43699
           1       0.43      0.65      0.52     15946

    accuracy                           0.68     59645
   macro avg       0.64      0.67      0.64     59645
weighted avg       0.73      0.68      0.70     59645

